# ⚙️ FASE 1 Lanjutan: Preprocessing & Feature Engineering
## Prediksi PM2.5 di Cekungan Bandung Menggunakan LSTM dengan Interpretasi SHAP
**Kerja Praktik — PRSDI BRIN 2026**

---

Notebook ini melanjutkan hasil dari Eksplorasi Data (EDA) dengan fokus menyiapkan data murni agar siap dimasukkan ke dalam arsitektur Machine Learning (XGBoost) dan Deep Learning (LSTM).

**Langkah-langkah utama di notebook ini:**
1. **Handling Missing Values:** Interpolasi nilai `NaN` pada PM2.5.
2. **Feature Engineering:** Ekstraksi fitur waktu siklik (Cyclical Time Features), penanda musim, dan variabel Lag (historis).
3. **Chronological Train-Test Split:** Membagi data secara berurutan waktu (80% Train, 20% Test).
4. **Data Normalization:** Penskalaan fitur (MinMaxScaler) yang wajib dilakukan sebelum training LSTM.


## 1. Import Library


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib  # Untuk menyimpan model Scaler
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11})
print("[OK] Library loaded.")


## 2. Load Dataset Merged (Kontinu)
Kita menggunakan data versi 3 (`merged_pm25_era5v3.csv`) yang berisi persis 1 tahun data berurutan (Nov 2022 - Nov 2023) dengan 9.480 baris.


In [ ]:
# Load data
data_path = r'D:\merged_pm25_era5v3.csv'
df = pd.read_csv(data_path)

# Rename kolom pertama menjadi datetime dan jadikan index
df.rename(columns={df.columns[0]: 'datetime'}, inplace=True)
df['datetime'] = pd.to_datetime(df['datetime'])
df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

print(f"Shape Dataset : {df.shape}")
print(f"Total Missing PM2.5 : {df['pm25'].isnull().sum()} baris")
df.head()


## 3. Handling Missing Values (Imputasi Time-Series)
Karena ini adalah data deret waktu, kita **DILARANG** menggunakan rata-rata global. 
Kita akan menggunakan **Interpolasi Linear** untuk menyambung garis antara titik sebelum sensor mati dan titik setelah sensor menyala kembali.


In [ ]:
# Visualisasi sebagian data yang berlubang sebelum imputasi
subset_gaps = df.loc['2023-04-10':'2023-05-10'].copy()

plt.figure(figsize=(15, 4))
plt.plot(subset_gaps.index, subset_gaps['pm25'], color='red', marker='.', linestyle='-', label='PM2.5 Asli (Banyak Gap)')
plt.title("Sebelum Interpolasi (Bulan April-Mei 2023)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Lakukan interpolasi (metode time/linear)
df['pm25'] = df['pm25'].interpolate(method='time')

# Pastikan sudah tidak ada NaN
print(f"Total Missing setelah interpolasi: {df.isnull().sum().sum()}")

# Visualisasi setelah imputasi
subset_fixed = df.loc['2023-04-10':'2023-05-10'].copy()
plt.figure(figsize=(15, 4))
plt.plot(subset_fixed.index, subset_fixed['pm25'], color='green', marker='.', linestyle='-', label='PM2.5 Setelah Interpolasi')
plt.title("Setelah Interpolasi (Garis tersambung mulus)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. Feature Engineering

Menambahkan variabel/kolom baru yang akan sangat membantu model LSTM memahami pola alam.

### 4.1 Ekstrak Fitur Waktu (Temporal Features)


In [ ]:
# Ambil info jam, hari, dan bulan
df['hour'] = df.index.hour
df['day_of_week'] = df.index.dayofweek
df['month'] = df.index.month

# Fitur Musim (Biner: 1=Kemarau, 0=Hujan)
# Kemarau di Bandung: Juni (6) sampai September (9)
df['is_dry_season'] = df['month'].apply(lambda x: 1 if x in [6, 7, 8, 9] else 0)

df[['hour', 'day_of_week', 'month', 'is_dry_season']].head()


### 4.2 Encoding Siklik (Cyclical Encoding)
Jam 23:00 dan jam 00:00 sebenarnya berdekatan (selisih 1 jam), tapi algoritma mengira selisihnya 23! 
Oleh karena itu, fitur waktu harus diubah ke dalam bentuk gelombang Sinus dan Cosinus.


In [ ]:
# Transformasi Siklik untuk Jam (24 Jam)
df['hour_sin'] = np.sin(df['hour'] * (2. * np.pi / 24))
df['hour_cos'] = np.cos(df['hour'] * (2. * np.pi / 24))

# Transformasi Siklik untuk Bulan (12 Bulan)
df['month_sin'] = np.sin((df['month']-1) * (2. * np.pi / 12))
df['month_cos'] = np.cos((df['month']-1) * (2. * np.pi / 12))

# Drop kolom original yang tidak diperlukan lagi untuk training
df.drop(columns=['hour', 'month'], inplace=True)


### 4.3 Fitur Historis (Lagged Features)
Untuk memprediksi PM2.5 di jam $T$, apa indikator terbaiknya? Tentu saja nilai PM2.5 di jam $T-1$ (satu jam yang lalu) dan $T-24$ (kemarin di jam yang sama).


In [ ]:
# Lag 1 Jam (PM2.5 satu jam yang lalu)
df['pm25_lag_1h'] = df['pm25'].shift(1)

# Lag 24 Jam (PM2.5 di jam yang sama kemarin)
df['pm25_lag_24h'] = df['pm25'].shift(24)

# Karena kita melakukan shift, 24 baris pertama akan memiliki nilai NaN.
# Kita drop 24 baris tersebut.
df.dropna(inplace=True)

print(f"Shape data setelah Feature Engineering: {df.shape}")
df[['pm25', 'pm25_lag_1h', 'pm25_lag_24h']].head()


## 5. Train - Test Split
Dalam Time-Series, kita **DILARANG** membagi data secara acak (`train_test_split(random_state=42)`). Kita harus membaginya secara kronologis (waktu).

- **Training Set (80%)**: 10 Bulan Pertama (Nov 2022 - Sep 2023)
- **Testing Set (20%)**: 2 Bulan Terakhir (Sep 2023 - Nov 2023)


In [ ]:
train_size = int(len(df) * 0.8)

train_df = df.iloc[:train_size].copy()
test_df = df.iloc[train_size:].copy()

print(f"Training Data : {len(train_df)} baris ({train_df.index.min()} sampai {train_df.index.max()})")
print(f"Testing Data  : {len(test_df)} baris ({test_df.index.min()} sampai {test_df.index.max()})")

plt.figure(figsize=(15, 4))
plt.plot(train_df.index, train_df['pm25'], label='Data Latih (Train)')
plt.plot(test_df.index, test_df['pm25'], label='Data Uji (Test)', color='orange')
plt.title('Chronological Train-Test Split (80% - 20%)')
plt.legend()
plt.show()


## 6. Normalisasi (Scaling)
Jaringan Saraf Tiruan (Deep Learning / LSTM) sangat sensitif terhadap skala data. Variabel "Suhu" rentangnya 20-30, sedangkan "Tekanan" rentangnya 800-900. Ini akan membuat model bingung.
Semua fitur harus ditekan ke rentang **0 hingga 1** menggunakan `MinMaxScaler`.


In [ ]:
# Inisialisasi Scaler
# Kita butuh 2 scaler:
# 1. scaler_X (untuk fitur cuaca dll)
# 2. scaler_y (khusus untuk PM2.5 agar gampang di-inverse nanti)

target_col = 'pm25'
feature_cols = [c for c in df.columns if c != target_col]

scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

# Fit dan Transform ke Data Train
X_train_scaled = scaler_X.fit_transform(train_df[feature_cols])
y_train_scaled = scaler_y.fit_transform(train_df[[target_col]])

# Transform Data Test (HANYA TRANSFORM, DILARANG FIT ke Data Test)
X_test_scaled = scaler_X.transform(test_df[feature_cols])
y_test_scaled = scaler_y.transform(test_df[[target_col]])

# Ubah kembali ke DataFrame agar rapi
train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=train_df.index)
train_scaled[target_col] = y_train_scaled

test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=test_df.index)
test_scaled[target_col] = y_test_scaled

print("Contoh data setelah di-scaling (semua rentang 0 sampai 1):")
display(train_scaled.head())


## 7. Simpan Data Siap Model
Data sudah bersih, sudah direkayasa, dan sudah diskala. Data ini akan dipanggil oleh Notebook XGBoost dan LSTM.


In [ ]:
OUTPUT_DIR = r'D:\Kuliah Praktik\KP BRIN\data\processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Simpan CSV
train_scaled.to_csv(os.path.join(OUTPUT_DIR, 'train_scaled.csv'))
test_scaled.to_csv(os.path.join(OUTPUT_DIR, 'test_scaled.csv'))

# Simpan Scaler (Penting untuk saat evaluasi membalikkan nilai dari 0-1 ke ug/m3)
joblib.dump(scaler_X, os.path.join(OUTPUT_DIR, 'scaler_X.pkl'))
joblib.dump(scaler_y, os.path.join(OUTPUT_DIR, 'scaler_y.pkl'))

print(f"[OK] Berhasil! Seluruh data dan Scaler telah disimpan di folder: {OUTPUT_DIR}")
print(f"-> train_scaled.csv")
print(f"-> test_scaled.csv")
print(f"-> scaler_X.pkl")
print(f"-> scaler_y.pkl")
